# Lesson 17: Checkpointing & Resume — Durable Graph Execution

## Objective
Use **checkpointers** to save graph state after every node, enabling **resume from any point** after a crash, interruption, or pause.

## Limitation of Lesson 16
- ❌ If the graph crashed mid-run, ALL progress was lost
- ❌ No way to "pick up where we left off"
- ❌ Multi-turn conversations needed manual state management

## What is NEW in Lesson 17?
- ✅ `MemorySaver` — in-memory checkpointer (development)
- ✅ `SqliteSaver` — SQLite-backed checkpointer (production)
- ✅ `thread_id` in config — identifies a conversation thread
- ✅ Automatic state persistence after every node
- ✅ Resume a conversation with the same `thread_id`
- ✅ `get_state()` — inspect saved checkpoint

## Theory: Checkpointing

```
Without checkpoint:
  run 1: node1 ✓ → node2 ✓ → node3 ✗ CRASH → lost everything

With checkpoint (MemorySaver):
  run 1: node1 ✓ [saved] → node2 ✓ [saved] → node3 ✗ CRASH
  run 2: resume from node2 checkpoint → node3 ✓ [saved] → END
```

### Thread ID
Every conversation gets a unique `thread_id`.  
The checkpointer stores state keyed by `(thread_id, checkpoint_id)`.

```python
config = {"configurable": {"thread_id": "session-001"}}
graph.invoke(state, config=config)     # Run 1 — saves checkpoints
graph.invoke(new_input, config=config) # Run 2 — resumes SAME thread
```

Calling `invoke()` on the SAME `thread_id` continues the conversation.  
This is how LangGraph enables multi-turn chat with automatic memory.


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Optional, Annotated
import operator
from IPython.display import Image, display

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

print("✓ Setup complete")
print("  MemorySaver: in-memory checkpointer (state persists between invoke() calls)")


In [ ]:
# ── Cell 2: State Schema ────────────────────────────────────────────────────
class ArithState(MessagesState):
    last_result: Optional[float]
    turn_count: int

print("✓ State defined (extends MessagesState for automatic message accumulation)")


In [ ]:
# ── Cell 3: Arithmetic Agent ─────────────────────────────────────────────────
SYSTEM = "You are an arithmetic assistant. Remember previous results. End with RESULT: <number>."

def agent_node(state: ArithState) -> dict:
    turn = state.get("turn_count", 0) + 1
    last = state.get("last_result")
    
    context = [SystemMessage(content=SYSTEM)]
    if last is not None:
        context.append(SystemMessage(content=f"Previous result: {last}"))
    context.extend(state["messages"])
    
    print(f"  [agent] Turn {turn} | {len(state['messages'])} messages in history")
    resp = llm.invoke(context)
    content = resp.content.strip()
    print(f"  [agent] Response: {content[:80]}")
    
    result = last
    for line in content.split("\n"):
        if "RESULT:" in line:
            try: result = float(line.split("RESULT:")[1].strip())
            except: pass
    
    return {
        "messages": [AIMessage(content=content)],
        "last_result": result,
        "turn_count": turn
    }

print("✓ Agent node defined")


In [ ]:
# ── Cell 4: Build Graph WITH Checkpointer ────────────────────────────────────
# The key change: pass checkpointer=... to compile()
# This makes the graph STATEFUL across invoke() calls on the same thread_id

memory = MemorySaver()   # In-memory checkpointer

builder = StateGraph(ArithState)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

# ── THE KEY LINE: pass the checkpointer at compile time ──────────────────
graph = builder.compile(checkpointer=memory)

print("✓ Graph compiled WITH MemorySaver checkpointer")
print("  Now: same thread_id = continuous conversation across multiple invoke() calls")


In [ ]:
# ── Cell 5: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 6: Multi-Turn Conversation (Same Thread) ────────────────────────────
# thread_id = "math-session-1" → all calls share state automatically

config = {"configurable": {"thread_id": "math-session-1"}}

def ask(question: str):
    print(f"\nUser: {question}")
    out = graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config   # Same thread_id = same conversation
    )
    print(f"Agent: {out['messages'][-1].content[:100]}")
    print(f"  [turn={out['turn_count']}, result={out['last_result']}]")
    return out

# These three calls share state — like a real conversation
ask("what is 25 plus 17?")
ask("multiply that by 3")
ask("subtract 10 from the result")


In [ ]:
# ── Cell 7: Inspect Saved Checkpoints ─────────────────────────────────────────
# get_state() retrieves the latest checkpoint for a thread

state_snapshot = graph.get_state(config)
print("Latest checkpoint for thread 'math-session-1':")
print(f"  Turn count: {state_snapshot.values.get('turn_count')}")
print(f"  Last result: {state_snapshot.values.get('last_result')}")
print(f"  Messages: {len(state_snapshot.values.get('messages', []))} messages")
print(f"  Next node: {state_snapshot.next}")

# List all checkpoints
print("\nAll checkpoints for this thread:")
for checkpoint in graph.get_state_history(config):
    print(f"  Step {checkpoint.metadata.get('step', '?')} | "
          f"next={checkpoint.next} | "
          f"turn={checkpoint.values.get('turn_count','?')}")


In [ ]:
# ── Cell 8: Different Thread = Fresh Session ─────────────────────────────────
# A different thread_id starts completely fresh
config2 = {"configurable": {"thread_id": "math-session-2"}}

print("New thread 'math-session-2' — fresh conversation:")
ask_in_thread = lambda q, cfg: graph.invoke(
    {"messages": [HumanMessage(content=q)]}, config=cfg
)

out = ask_in_thread("what is 7 times 8?", config2)
print(f"  Thread 2 result: {out['last_result']}")
print(f"  Thread 2 turn:   {out['turn_count']}")

# Original thread still remembers its own history
out_orig = ask_in_thread("what was the last result?", config)
print(f"\n  Thread 1 still remembers: turn={out_orig['turn_count']}, result={out_orig['last_result']}")


In [ ]:
# ── Cell 9: SqliteSaver for Production ───────────────────────────────────────
# MemorySaver is lost when process restarts
# SqliteSaver persists to disk — survives restarts

from langgraph.checkpoint.sqlite import SqliteSaver

CHECKPOINT_DB = "/tmp/arithmetic_checkpoints.db"
sqlite_checkpointer = SqliteSaver.from_conn_string(CHECKPOINT_DB)

graph_persistent = builder.compile(checkpointer=sqlite_checkpointer)

config3 = {"configurable": {"thread_id": "persistent-session-1"}}

print("Testing SqliteSaver (persists to disk):")
out = graph_persistent.invoke(
    {"messages": [HumanMessage(content="what is 99 divided by 9?")]},
    config=config3
)
print(f"  Result: {out['last_result']}")
print(f"  Saved to: {CHECKPOINT_DB}")
print("  This checkpoint survives process restart!")


## Checkpointer Architecture

```
graph.invoke(input, config={"configurable": {"thread_id": "abc"}})
                                                        ↓
                                               ┌─────────────────┐
                                               │  Checkpointer   │
                                               │  (Memory/SQLite) │
                                               │                 │
                                               │ thread "abc"    │
                                               │  step 0: {...}  │
                                               │  step 1: {...}  │
                                               │  step 2: {...}  │
                                               └─────────────────┘
```

### Available Checkpointers

| Checkpointer | Use Case | Persistence |
|-------------|---------|-------------|
| `MemorySaver` | Development, testing | Process lifetime |
| `SqliteSaver` | Single-server production | Disk (SQLite file) |
| `PostgresSaver` | Multi-server production | PostgreSQL database |
| `RedisSaver` | High-throughput | Redis |

## Summary

| | Without Checkpoint | With Checkpoint |
|--|--------------------|----------------|
| Multi-turn | Manual state passing | Automatic via thread_id |
| Crash recovery | Lost | Resume from last step |
| State inspection | None | `get_state()`, `get_state_history()` |

## Limitations → What Lesson 18 Solves
- ❌ The graph runs to completion without stopping for human review
- ❌ Critical operations (like large multiplications) proceed without approval
- ❌ **Lesson 18**: Human-in-the-loop — pause execution, wait for human approval, resume
